In [1]:
#=================================
# Librerias
#=================================
import pennylane as qml
import numpy as np
from scipy.integrate import quad

In [2]:
#========================================================
# QME Mejorado con Preparación de Estado Consciente
#========================================================

class QuantumMeanEstimator:
    """
    Clase que implementa el algoritmo QME con preparación 
    de estado consciente del integrando.
    """
    
    def __init__(self, n_qubits, a, b, f_func, f_max):
        """
        Inicializar el estimador cuántico.
        
        Args:
            n_qubits: Número de qubits de índice
            a, b: Límites del intervalo de integración
            f_func: Función integrando f(x)
            f_max: Valor máximo del integrando
        """
        self.n_qubits = n_qubits
        self.N = 2**n_qubits
        self.a = a
        self.b = b
        self.f_func = f_func
        self.f_max = f_max
        
        # Crear malla de discretización
        self.x_values = np.linspace(a, b, self.N)
        self.f_values = np.array([f_func(x) for x in self.x_values])
        self.f_norm = self.f_values / f_max
        
        # Crear dispositivo
        self.dev = qml.device("default.qubit", wires=n_qubits+1)
    
    @property
    def circuit(self):
        """Retorna el circuito QME compilado."""
        @qml.qnode(self.dev)
        def _circuit():
            # Preparar superposición uniforme
            for i in range(self.n_qubits):
                qml.Hadamard(wires=i)
            
            # Codificación del integrando mediante
            # rotaciones RY parametrizadas
            for i in range(self.n_qubits):
                for k in range(2**i):
                    idx = k * 2**(self.n_qubits - i)
                    if idx < len(self.f_norm):
                        theta = 2 * np.arcsin(self.f_norm[idx])
                        qml.RY(theta, wires=self.n_qubits)
            
            # Medir el observable Z en el qubit ancila
            return qml.expval(qml.PauliZ(self.n_qubits))
        
        return _circuit
    
    def estimate_integral(self):
        """
        Estimar la integral utilizando el circuito QME.
        
        Returns:
            Valor estimado de la integral
        """
        expectation_z = self.circuit()
        integral = (self.b - self.a) * self.f_max * expectation_z / self.N
        return integral
    
    def exact_integral(self):
        """Calcular la integral exacta para comparación."""
        from scipy.integrate import quad
        result, _ = quad(self.f_func, self.a, self.b)
        return result

In [3]:
#========================================================
# Uso de la Clase QuantumMeanEstimator
#========================================================

# Definir la función integrando
p = 4
def f(x):
    return x**p

# Parámetros
n_qubits = 8
a, b = -1.0, 1.0
f_max = np.max(np.abs(np.linspace(a, b, 2**n_qubits)**p))

# Crear estimador cuántico
estimator = QuantumMeanEstimator(n_qubits, a, b, f, f_max)

# Estimar la integral
integral_qme = estimator.estimate_integral()
integral_exact = estimator.exact_integral()

# Mostrar resultados
print("ALGORITMO QME - VERSIÓN MEJORADA")
print(f"Integral estimada: {integral_qme:.6f}")
print(f"Integral exacta:   {integral_exact:.6f}")
print(f"Error relativo:    {np.abs(integral_qme - integral_exact)/np.abs(integral_exact)*100:.4f}%")

ALGORITMO QME - VERSIÓN MEJORADA
Integral estimada: 0.003111
Integral exacta:   0.400000
Error relativo:    99.2222%
